# Section 4 — Companion Notebook

**Paper:** *Birth-Death Stochastic Conic Particle Gradient Descent for global optimization on the space of measures*
(De Castro, Gadat & Marteau, 2026 — JMLR submission)

This notebook is the companion to **Section 4 (Numerical experiments)** of the paper. It walks through the two experiments — Gaussian Mixture density estimation (§4.1) and a two-layer ReLU network on California Housing (§4.2) — and reproduces all the figures and tables of §4.3 from a previously saved set of result artefacts.

> Each markdown cell names the paper section it mirrors. Each code cell either runs a ***Light demo*** (a small-budget version of the full experiment, ~2–3 min total) or **loads a saved paper figure** from a `results_*` directory.


## How to read this notebook

| Convention | Meaning |
|-----------|---------|
| **Light demo** | A scaled-down re-run of the experiment so the math comes alive. Numbers will differ from the paper. |
| **Paper figure** | A PDF artefact loaded from a saved `results_*` directory, identical to what appears in Section 4. |
| **Math appendix** | A self-contained derivation cross-referenced to the paper's Section 2/3 or appendices. |

**Source layout.**
- Saved GMM run: `results_gmm_fixed_cov_20260305_115544/` (matches the paper's Figs. 1–4 and Table 2).
- Saved NN run: `results_nn_california_20260303_171130/` (matches the paper's Figs. 2–3 top row and Table 3; the extra `fig3_dual_cert_*`, `fig4_weights_*`, `fig5_positions_*` panels are companion-only pedagogical figures, not in the paper).
- Paper figures: `../figures/`.

### Table of contents (mirrors §4)

1. **Shared math & code** — BLASSO recap, CPGD update, Birth & Death rules (cross-refs to §2–§3).
2. **§4.1 GMM with Fixed Covariance** — problem, kernel derivation, demo, paper figures.
3. **§4.2 Two-layer NN on California Housing** — problem, ReLU kernel, demo, paper figures.
4. **§4.3 Experimental Results & Discussion** — §4.3.1 BD dynamics, §4.3.2 convergence, §4.3.3 spatial distribution, §4.3.4 performance tables.
5. **Math appendix** — symmetrization, mirror retraction, dual certificate, BD score derivation.


In [ ]:
import json, os, time, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Any, Callable, TypeVar, Generic, ClassVar
from abc import ABC, abstractmethod
from functools import cached_property

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import Tensor
from torch.nn import Module, Parameter
from jaxtyping import Float, Int

from IPython.display import IFrame, Markdown, display, HTML

# Reproducibility
torch.manual_seed(314)
np.random.seed(314)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths (relative to notebook_python/)
PAPER_FIG_DIR = Path("../figures")
GMM_RESULTS   = Path("results_gmm_fixed_cov_20260305_115544")
NN_RESULTS    = Path("results_nn_california_20260303_171130")

def show_pdf(path, width=560, height=380, caption=None):
    """Embed a PDF figure inline. Falls back to a link if rendering fails."""
    p = Path(path)
    if caption:
        display(Markdown(f"**{caption}**"))
    if not p.exists():
        display(Markdown(f"⚠️  Missing figure: `{p}` — run the upstream notebooks to regenerate."))
        return
    try:
        display(IFrame(str(p), width=width, height=height))
    except Exception:
        display(Markdown(f"[Open `{p}`]({p})"))

def show_pdf_grid(paths_with_captions, ncols=2, width=420, height=300):
    """Layout multiple PDFs in a grid via HTML."""
    cells_html = []
    for path, cap in paths_with_captions:
        p = Path(path)
        if not p.exists():
            cells_html.append(f"<td style='padding:6px;border:1px solid #eee;'><b>{cap}</b><br/>⚠️ missing: {p}</td>")
        else:
            cells_html.append(
                f"<td style='padding:6px;border:1px solid #eee;vertical-align:top;'>"
                f"<b>{cap}</b><br/>"
                f"<iframe src='{p}' width='{width}' height='{height}'></iframe></td>"
            )
    rows = ["<tr>" + "".join(cells_html[i:i+ncols]) + "</tr>" for i in range(0, len(cells_html), ncols)]
    display(HTML("<table>" + "".join(rows) + "</table>"))

print("Paths:")
print(f"  paper figures : {PAPER_FIG_DIR.resolve()}")
print(f"  GMM artefacts : {GMM_RESULTS.resolve()}")
print(f"  NN  artefacts : {NN_RESULTS.resolve()}")


## 1. Shared math & code

### 1.1 BLASSO recap 

The two experiments minimise the same Beurling LASSO objective over the space of finite positive measures $\mathcal{M}_+(\mathcal{X})$:
$$
J(\nu) \;=\; \tfrac{1}{2}\,\|y - \Phi(\nu)\|_{\mathbb{H}}^{2} \;+\; \kappa\,\|\nu\|_{\mathrm{TV}},\qquad \Phi(\nu)=\int \varphi_t\,\mathrm{d}\nu(t),
$$
with feature map $\varphi:\mathcal{X}\to\mathbb{H}$ and TV norm $\|\nu\|_{\mathrm{TV}}=\nu(\mathcal{X})$. The two instantiations differ only in the choice of $(\mathcal{X},\varphi,y,\mathbb{H})$:
- **GMM (§4.1)** — $\mathcal{X}=\mathbb{R}^{2}$, $\varphi_t = \mathcal{N}(\cdot;t,(1+\tau^{2})I_{2})$, $y=\widehat{f}_{\tau}^{\,n}$, $\mathbb{H}=L^{2}(\mathbb{R}^{2})$.
- **NN (§4.2)** — $\mathcal{X}=\bar{B}(0,1)\subset\mathbb{R}^{d+1}$, $\varphi_{(w,b)}(x)=\mathrm{ReLU}(\langle w,x\rangle+b)$, $y$ the regression target, $\mathbb{H}=\mathbb{R}^{n}$ with normalised inner product.

### 1.2 CPGD update 

For a discrete measure $\nu = \sum_{j} \omega_j \delta_{t_j}$, the Fréchet derivative reads
$$
J'_\nu(t)\;=\;\kappa\;+\;\sum_{j}\omega_j\,K(t,t_j)\;-\;\langle\varphi_t,\,y\rangle_{\mathbb{H}}.
$$
The Weight & Push-Forward update of §2.3 acts as
$$
\omega_j \leftarrow \omega_j\,e^{-\alpha\,J'_\nu(t_j)},\qquad t_j \leftarrow t_j - \beta\,\nabla_t J'_\nu(t_j),
$$
with step sizes $\alpha,\beta>0$. For the GMM experiment we use a reparametrisation $\omega_j=r_j^{2}$ and Euclidean retraction; for the NN experiment we use mirror retraction directly on $\omega_j>0$ and a projection onto $\bar{B}(0,1)$.


### 1.3 Birth & Death rules 

Both experiments use the same family of rules.

**Death.** Compute, for each support point $t_j$, the score
$$
s_j \;=\; \frac{2\,J'_\nu(t_j)}{\omega_j\,K(t_j,t_j)}
$$
and remove the particle achieving $s_{j^\star}=\max_j s_j$ whenever $s_{j^\star}>\tau_{\mathrm{death}}$. The threshold is set to $\tau_{\mathrm{death}}=5$ in Section 4 (paper Table 1).

**Birth.** Sample $n_{\mathrm{test}}$ candidate positions, evaluate $J'_\nu$ at each, and spawn a new particle at the argmin position whenever
$$
\min_k J'_\nu(\widehat t^{(k)})\;<\;\tau_{\mathrm{birth}}\,\sqrt{\frac{\log m_k}{m_k}}
$$
with mini-batch size $m_k$. The full-batch regime uses a negative $\tau_{\mathrm{birth}}$ (genuine KKT violations only); the stochastic regime uses a positive $\tau_{\mathrm{birth}}$ to account for mini-batch noise.

A derivation of the death score is given in the math appendix at the end of the notebook.


In [ ]:
# ============================================================================
# RETRACTION CLASSES
# ============================================================================

class Retraction:
    """Euclidean retraction (additive update)."""
    def __call__(self, positions, weights, dpositions, dweights):
        positions += dpositions
        weights   += dweights

class MirrorRetraction(Retraction):
    """Mirror retraction (multiplicative update on weights)."""
    def __call__(self, positions, weights, dpositions, dweights):
        positions += dpositions
        weights   *= torch.exp(dweights)


# ============================================================================
# PARTICLES
# ============================================================================

class Particles(Module):
    """Discrete measure mu_p = sum_i eps_i * h(r_i) * delta_{theta_i}."""

    def __init__(self, h, positions, weights, signs):
        n, _ = positions.shape
        assert weights.shape == (n,)
        assert signs.shape   == (n,)
        super().__init__()
        self.h = h
        self.positions = Parameter(positions)
        self.weights   = Parameter(weights)
        self.signs     = signs
        self.gradient_evaluation = None

    @property
    def amplitudes(self):
        return self.h(self.weights)

    @property
    def amplitudes_deriv(self):
        with torch.enable_grad():
            a = self.amplitudes
            return torch.autograd.grad(a, self.weights, grad_outputs=torch.ones_like(a))[0]

    def __len__(self):
        return self.positions.size(0)

    def tv_norm(self):
        return self.amplitudes.abs().sum()


# ============================================================================
# BLASSO PROBLEMS (abstract + L2)
# ============================================================================

TProblem = TypeVar("TProblem", bound="BlassoProblem")

@dataclass
class ModelEvaluation(Generic[TProblem]):
    problem: TProblem
    particles: Particles

    def set_particles_gradient(self, dweights, dpositions):
        self.particles.weights.grad   = dweights
        self.particles.positions.grad = dpositions
        self.particles.gradient_evaluation = self

    @cached_property
    def proximity(self):
        raise NotImplementedError

    @cached_property
    def lasso(self):
        return self.problem.kappa * self.particles.tv_norm()

    @cached_property
    def loss(self):
        return self.proximity + self.lasso

    @cached_property
    def Fm_backward(self):
        dw, dp = torch.autograd.grad(self.loss, [self.particles.weights, self.particles.positions])
        self.set_particles_gradient(dw, dp)

    @cached_property
    def Jp(self):
        raise NotImplementedError

    @cached_property
    def loss_backward(self):
        try:
            self.Jp_backward
        except NotImplementedError:
            self.Fm_backward

    def backward(self):
        self.loss_backward

    def item(self):
        return self.loss.item()


@dataclass
class BlassoProblem(ABC):
    Evaluation: ClassVar[type] = ModelEvaluation
    kappa: float
    def __call__(self, particles):
        return self.Evaluation(self, particles)


Observation = TypeVar("Observation")

@dataclass
class L2Blasso(BlassoProblem, Generic[Observation]):
    y: Observation
    loss_offset: float = field(init=False, default=0.0)

    @dataclass
    class Evaluation(ModelEvaluation["L2Blasso"]):
        @cached_property
        def kernel(self):
            raise NotImplementedError
        @cached_property
        def scalar_products(self):
            raise NotImplementedError
        @cached_property
        def l2_cost(self):
            a = self.particles.amplitudes
            K, S = self.kernel, self.scalar_products
            return 0.5 * a @ (K @ a) - a @ S + self.problem.loss_offset
        @cached_property
        def proximity(self):
            return self.l2_cost


print("Base classes (Particles, ModelEvaluation, BlassoProblem, L2Blasso, retractions) loaded.")


## 2. §4.1 Gaussian Mixture Model with Fixed Covariance

We observe $n$ i.i.d. samples $X_1,\dots,X_n\in\mathbb{R}^{2}$ from a 2D Gaussian mixture
$$
f^{0}=\sum_{k=1}^{K} a_k^{0}\,\mathcal{N}(\cdot;\theta_k^{0},I_2),\qquad K=25.
$$
With smoothing kernel $\phi_\tau=\mathcal{N}(0,\tau^{2}I_{2})$ and feature map $\varphi_t(x)=\mathcal{N}(x;t,(1+\tau^{2})I_{2})$, the regularised $L^{2}$ objective (paper Eq. (4.1)) reads
$$
J(\nu)\;=\;\tfrac{1}{2}\,\|\phi_\tau * \nu - \widehat f_\tau^{\,n}\|_{L^{2}(\mathbb{R}^{2})}^{2}\;+\;\kappa\,\|\nu\|_{\mathrm{TV}}.
$$
The kernel $K(t_i,t_j)=\mathcal{N}(t_i;t_j,2(1+\tau^{2})I_{2})$ is bounded, $\mathcal{C}^{\infty}$ and translation-invariant, so the assumption $(\mathcal{H}_{\mathcal{P}})$ of §2.2 holds exactly.


### 2.1 Kernel and scalar-products derivation

For $\sigma^{2}=1+\tau^{2}$, the convolution of two Gaussians yields
$$
K(t_i,t_j)\;=\;\langle\varphi_{t_i},\varphi_{t_j}\rangle_{L^{2}}\;=\;\mathcal{N}(t_i;t_j,2\sigma^{2}I_{2})\;=\;\frac{1}{4\pi\sigma^{2}}\exp\!\left(-\frac{\|t_i-t_j\|^{2}}{4\sigma^{2}}\right),
$$
and the self-kernel $K(t,t)=1/(4\pi\sigma^{2})$ is **constant** in $t$ — this is what makes the GMM death score (§1.3) particularly simple.

For the data side, with $\widehat f_\tau^{\,n}=\frac{1}{n}\sum_{i}\phi_\tau(\cdot-X_i)$,
$$
S(t)\;=\;\langle\widehat f_\tau^{\,n},\varphi_t\rangle_{L^{2}}\;=\;\frac{1}{n}\sum_{i=1}^{n}\mathcal{N}(X_i;t,(1+2\tau^{2})I_{2}).
$$
The Fréchet derivative used by CPGD and by the BD rules is then $J'_\nu(t)=\kappa+(Ka)(t)-S(t)$, a quantity that admits a closed form at any test point $t$ (used by `compute_jp_gmm` below).


In [ ]:
# ============================================================================
# GMM-specific classes and helpers
# ============================================================================

def generate_gmm_data(means, weights, n_samples, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    probs = weights / weights.sum()
    component_idx = torch.multinomial(probs, n_samples, replacement=True)
    return means[component_idx] + torch.randn(n_samples, 2)


class FixedCovGMMParticles(Particles):
    """Particles with h(r)=r^2, signs all +1, positions in R^2."""
    def __init__(self, positions, weights):
        n = positions.shape[0]
        h = lambda r: r**2
        signs = torch.ones(n)
        super().__init__(h=h, positions=positions, weights=weights, signs=signs)

    def clone(self, *, detach=True):
        pos = self.positions.detach().clone() if detach else self.positions.clone()
        w   = self.weights.detach().clone()   if detach else self.weights.clone()
        return FixedCovGMMParticles(pos, w)


@dataclass
class FixedCovGMMBlasso(L2Blasso[Tensor]):
    tau: float
    skip_offset: bool = False
    d: int = field(init=False, default=2)

    def __post_init__(self):
        n, self.d = self.y.shape
        if self.skip_offset:
            self.loss_offset = 0.0
            return
        # diagonal approximation of ||hat_f_tau||^2 (constant, drops out of gradients)
        self.loss_offset = float(1.0 / (2 * n * (4 * math.pi * self.tau**2)))

    @dataclass
    class Evaluation(L2Blasso.Evaluation):
        @cached_property
        def _sigma2_k(self):  return 2.0 * (1.0 + self.problem.tau**2)
        @cached_property
        def _sigma2_s(self):  return 1.0 + 2.0 * self.problem.tau**2

        @cached_property
        def kernel(self):
            theta = self.particles.positions
            s2, d = self._sigma2_k, 2
            diff = theta[:, None, :] - theta[None, :, :]
            sq   = (diff**2).sum(-1)
            return torch.exp(-sq / (2*s2) - (d/2)*math.log(2*math.pi*s2))

        @cached_property
        def scalar_products(self):
            X, theta = self.problem.y, self.particles.positions
            s2, d = self._sigma2_s, 2
            diff = theta[:, None, :] - X[None, :, :]
            sq   = (diff**2).sum(-1)
            log_phi = -sq / (2*s2) - (d/2)*math.log(2*math.pi*s2)
            return torch.exp(log_phi).mean(dim=1)


def _jp_method(self):
    a = self.particles.amplitudes
    return self.problem.kappa + self.kernel @ a - self.scalar_products
FixedCovGMMBlasso.Evaluation.Jp = cached_property(_jp_method)


def compute_jp_gmm(problem, particles, test_pos):
    """Analytical J'_nu(t) = kappa + (Ka)(t) - S(t) at arbitrary positions."""
    tau, d, kappa = problem.tau, 2, problem.kappa
    s2k, s2s = 2.0*(1.0+tau**2), 1.0+2.0*tau**2
    with torch.no_grad():
        a, theta = particles.amplitudes, particles.positions.data
        diff_k = test_pos[:, None, :] - theta[None, :, :]
        sq_k   = (diff_k**2).sum(-1)
        Kt     = torch.exp(-sq_k/(2*s2k) - (d/2)*math.log(2*math.pi*s2k))
        diff_s = test_pos[:, None, :] - problem.y[None, :, :]
        sq_s   = (diff_s**2).sum(-1)
        St     = torch.exp(-sq_s/(2*s2s) - (d/2)*math.log(2*math.pi*s2s)).mean(-1)
    return kappa + Kt @ a - St


# ---- GMM optimisers ---------------------------------------------------------

@dataclass
class GMMOptimizer:
    """Full-batch CPGD for h(r)=r^2."""
    particles: FixedCovGMMParticles
    problem:   FixedCovGMMBlasso
    beta:      float
    alpha:     float
    min_weight: float = 1e-6

    def step(self):
        ev = self.problem(self.particles); ev.backward()
        w, p = self.particles.weights, self.particles.positions
        a, ad = self.particles.amplitudes, self.particles.amplitudes_deriv
        with torch.no_grad():
            sd  = ad.abs().clamp(min=1e-10)
            sa  = a.clamp(min=1e-10)
            w.sub_(self.alpha * w.grad / sd); w.clamp_(min=self.min_weight)
            p.sub_(self.beta / sa.unsqueeze(-1) * p.grad)
        return ev.loss.item()

    def zero_grad(self): self.particles.zero_grad()


@dataclass
class StochasticGMMOptimizer:
    particles:  FixedCovGMMParticles
    problem:    FixedCovGMMBlasso
    beta:       float
    alpha:      float
    batch_size: int
    min_weight: float = 1e-6

    def __post_init__(self):
        self.n_train = len(self.problem.y)

    def step(self):
        idx = torch.randperm(self.n_train)[:self.batch_size]
        mini = FixedCovGMMBlasso(kappa=self.problem.kappa,
                                 y=self.problem.y[idx],
                                 tau=self.problem.tau,
                                 skip_offset=True)
        ev = mini(self.particles); ev.backward()
        w, p = self.particles.weights, self.particles.positions
        a, ad = self.particles.amplitudes, self.particles.amplitudes_deriv
        with torch.no_grad():
            sd, sa = ad.abs().clamp(min=1e-10), a.clamp(min=1e-10)
            w.sub_(self.alpha * w.grad / sd); w.clamp_(min=self.min_weight)
            p.sub_(self.beta / sa.unsqueeze(-1) * p.grad)
        return ev.loss.item()

    def zero_grad(self): self.particles.zero_grad()


# ---- GMM Birth–Death --------------------------------------------------------

def _bd_remove_gmm(particles, idx):
    keep = [i for i in range(len(particles)) if i != idx]
    with torch.no_grad():
        return FixedCovGMMParticles(particles.positions.data[keep].clone(),
                                    particles.weights.data[keep].clone())

def _bd_add_gmm(particles, new_pos, new_w):
    with torch.no_grad():
        pos = torch.cat([particles.positions.data, new_pos.unsqueeze(0)], 0)
        w   = torch.cat([particles.weights.data,   torch.tensor([new_w])], 0)
    return FixedCovGMMParticles(pos, w)

def death_step_gmm(particles, problem, threshold):
    with torch.no_grad():
        tau = problem.tau
        K_self = 1.0 / (4 * math.pi * (1 + tau**2))
        jp = compute_jp_gmm(problem, particles, particles.positions.data)
        a  = particles.amplitudes.data
        scores = 2.0 * jp / (a.clamp(min=1e-10) * K_self)
        mv, mi = scores.max(0)
        if mv.item() > threshold:
            return _bd_remove_gmm(particles, mi.item()), True
    return particles, False

def birth_step_gmm(particles, problem, n_test, threshold, search_sigma):
    with torch.no_grad():
        tau = problem.tau
        K_self = 1.0 / (4 * math.pi * (1 + tau**2))
        test_pos = search_sigma * torch.randn(n_test, 2)
        jp_test  = compute_jp_gmm(problem, particles, test_pos)
        mv, mi = jp_test.min(0)
        if mv.item() < threshold:
            w0 = float(np.sqrt(max(mv.abs().item() / max(K_self, 1e-10), 1e-6)))
            return _bd_add_gmm(particles, test_pos[mi.item()], w0), True
    return particles, False

print("GMM classes, optimisers and BD helpers loaded.")


### 2.2 Data generation — paper geometry

Geometry copied from `experiments_fastpart_gmm_fixed_covariance.ipynb` ([cell 7](experiments_fastpart_gmm_fixed_covariance.ipynb)):
- $K=25$ component means on a circle of radius $R=30$;
- mixture weights drawn from $\mathrm{Dirichlet}(\mathbf{1}_{25})$, seed `31415`;
- $n=30\,000$ samples, then a $80/20$ split → **$24\,000$ train** + $6\,000$ test (matches the paper's Table).


In [ ]:
# ---- Ground-truth GMM -------------------------------------------------------
K_TRUE             = 25
CIRCLE_RADIUS      = 30.0
DIRICHLET_CONC     = 1.0

angles      = torch.linspace(0, 2*math.pi, K_TRUE + 1)[:-1]
true_means  = CIRCLE_RADIUS * torch.stack([torch.cos(angles), torch.sin(angles)], 1)
torch.manual_seed(31415)
true_weights = torch.distributions.Dirichlet(DIRICHLET_CONC * torch.ones(K_TRUE)).sample()

# Observations
N_SAMPLES = 30_000
TAU_GMM   = 0.1
X_all     = generate_gmm_data(true_means, true_weights, N_SAMPLES, seed=42)

# 80/20 train/test
torch.manual_seed(0)
n_test  = N_SAMPLES // 5
perm    = torch.randperm(N_SAMPLES)
X_train_gmm = X_all[perm[n_test:]]
X_test_gmm  = X_all[perm[:n_test]]

print(f"GMM: K={K_TRUE}, R={CIRCLE_RADIUS}, n={N_SAMPLES} -> "
      f"train={len(X_train_gmm)}, test={len(X_test_gmm)}")


In [ ]:
# ---- Data visualisation -----------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_all[:, 0], X_all[:, 1], s=5, alpha=0.25, color="steelblue", label="Observations")
m_np   = true_means.numpy()
w_norm = (true_weights / true_weights.sum()).numpy()
ax.scatter(m_np[:, 0], m_np[:, 1], s=20 + 2000*w_norm, marker="*",
           color="crimson", zorder=5, label=r"True means (size $\propto$ weight)")
ax.set_aspect("equal"); ax.legend()
ax.set_title(f"2D GMM, $n={N_SAMPLES}$, $K={K_TRUE}$, $R={CIRCLE_RADIUS}$")
plt.tight_layout(); plt.show()


### 2.3 Hyperparameters (paper Table 1, GMM column)

| Symbol | Paper value | Light-demo value |
|---|---|---|
| $n$ (train) | $24\,000$ | $24\,000$ |
| $p_{\mathrm{init}}$ | $20$ | $20$ |
| $\kappa$ | $10^{-4}$ | $10^{-4}$ |
| $\tau$ | $0.1$ | $0.1$ |
| Full-batch iterations | $50\,000$ | $5\,000$ |
| Stochastic batch $B$ | $256$ | $256$ |
| Stochastic iterations | $200\,000$ | $20\,000$ |
| $\tau_{\mathrm{death}}$ | $5$ | $5$ |
| $\eta$ (position) / $\eta_w$ (weight) | $0.5$ / $0.5$ | $0.5$ / $0.5$ |


In [ ]:
# ---- Hyperparameters --------------------------------------------------------
GMM_KAPPA        = 1e-4
GMM_N_PARTICLES  = 20
GMM_ETA          = 0.5
GMM_ETA_W        = 0.5
GMM_BATCH_SIZE   = 256

# Light demo budget (paper values: 50_000 / 200_000)
GMM_T_FULL       = 5_000
GMM_T_STO        = 20_000
GMM_LOG          = 200

# BD schedule (rescaled for demo budget)
BD_GMM_FULL_DEATH_DELAY = 1_000
BD_GMM_FULL_DEATH_EVERY = 200
BD_GMM_FULL_BIRTH_DELAY = 500
BD_GMM_FULL_BIRTH_EVERY = 250
BD_GMM_STO_DEATH_DELAY  = 4_000
BD_GMM_STO_DEATH_EVERY  = 500
BD_GMM_STO_BIRTH_DELAY  = 2_000
BD_GMM_STO_BIRTH_EVERY  = 1_000
BD_GMM_N_TEST           = 1_000

BD_TAU_DEATH         = 5.0
BD_GMM_TAU_BIRTH_FULL = -0.1 * math.sqrt(2 * math.log(len(X_train_gmm)) / len(X_train_gmm))
BD_GMM_TAU_BIRTH_STO  =  5.0 * math.sqrt(2 * math.log(GMM_BATCH_SIZE)   / GMM_BATCH_SIZE)

gmm_problem      = FixedCovGMMBlasso(kappa=GMM_KAPPA, y=X_train_gmm, tau=TAU_GMM)
gmm_problem_test = FixedCovGMMBlasso(kappa=GMM_KAPPA, y=X_test_gmm,  tau=TAU_GMM)


### 2.4 Light demo — four methods together

Runs Full-Batch / Stochastic / Full-Batch+BD / Stochastic+BD with the demo budget. Expect about **30 s** total on a CPU. Results will differ quantitatively from the paper (see §4.3.4 below for the paper numbers).


In [ ]:
def init_gmm_particles(n, sigma, seed):
    torch.manual_seed(seed)
    pos = sigma * torch.randn(n, 2)
    wts = (1.0 / n) * torch.ones(n)
    return FixedCovGMMParticles(pos, wts)


def run_gmm_method(method, T):
    p0 = init_gmm_particles(GMM_N_PARTICLES, 0.5*CIRCLE_RADIUS, seed=314)
    if method == "full":
        opt = GMMOptimizer(p0, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W)
        bd  = None
    elif method == "sto":
        opt = StochasticGMMOptimizer(p0, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W,
                                     batch_size=GMM_BATCH_SIZE)
        bd  = None
    elif method == "full_bd":
        opt = GMMOptimizer(p0, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W)
        bd  = dict(kind="full", death_delay=BD_GMM_FULL_DEATH_DELAY, death_every=BD_GMM_FULL_DEATH_EVERY,
                   birth_delay=BD_GMM_FULL_BIRTH_DELAY, birth_every=BD_GMM_FULL_BIRTH_EVERY,
                   tau_birth=BD_GMM_TAU_BIRTH_FULL)
    elif method == "sto_bd":
        opt = StochasticGMMOptimizer(p0, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W,
                                     batch_size=GMM_BATCH_SIZE)
        bd  = dict(kind="sto",  death_delay=BD_GMM_STO_DEATH_DELAY, death_every=BD_GMM_STO_DEATH_EVERY,
                   birth_delay=BD_GMM_STO_BIRTH_DELAY, birth_every=BD_GMM_STO_BIRTH_EVERY,
                   tau_birth=BD_GMM_TAU_BIRTH_STO)
    losses, tvs, times, ps, iters = [], [], [], [], []
    deaths, births = [], []
    t0 = time.time()
    for it in range(T):
        opt.zero_grad(); opt.step()
        if bd is not None:
            if it >= bd["death_delay"] and it % bd["death_every"] == 0 and len(opt.particles) > 1:
                newp, died = death_step_gmm(opt.particles, gmm_problem, BD_TAU_DEATH)
                if died:
                    deaths.append((it, len(newp)))
                    if bd["kind"] == "full":
                        opt = GMMOptimizer(newp, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W)
                    else:
                        opt = StochasticGMMOptimizer(newp, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W,
                                                     batch_size=GMM_BATCH_SIZE)
            if it >= bd["birth_delay"] and it % bd["birth_every"] == 0:
                newp, born = birth_step_gmm(opt.particles, gmm_problem,
                                            BD_GMM_N_TEST, bd["tau_birth"], 0.5*CIRCLE_RADIUS)
                if born:
                    births.append((it, len(newp)))
                    if bd["kind"] == "full":
                        opt = GMMOptimizer(newp, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W)
                    else:
                        opt = StochasticGMMOptimizer(newp, gmm_problem, beta=GMM_ETA, alpha=GMM_ETA_W,
                                                     batch_size=GMM_BATCH_SIZE)
        if it % GMM_LOG == 0 or it == T-1:
            with torch.no_grad():
                ev = gmm_problem(opt.particles)
                losses.append(ev.loss.item())
                tvs.append(opt.particles.tv_norm().item())
                ps.append(len(opt.particles))
                iters.append(it)
                times.append(time.time() - t0)
    return dict(particles=opt.particles, losses=losses, tvs=tvs, times=times,
                ps=ps, iters=iters, deaths=deaths, births=births, T=T)


print("Running GMM light demo (4 methods)...")
t_start = time.time()
gmm_full     = run_gmm_method("full",    GMM_T_FULL)
gmm_sto      = run_gmm_method("sto",     GMM_T_STO)
gmm_full_bd  = run_gmm_method("full_bd", GMM_T_FULL)
gmm_sto_bd   = run_gmm_method("sto_bd",  GMM_T_STO)
print(f"\nGMM demo finished in {time.time()-t_start:.1f}s.")

for name, h in [("Full", gmm_full), ("Sto", gmm_sto), ("Full+BD", gmm_full_bd), ("Sto+BD", gmm_sto_bd)]:
    print(f"  {name:<10s}  loss={h['losses'][-1]:.5f}  TV={h['tvs'][-1]:.3f}  "
          f"p_final={h['ps'][-1]:3d}  deaths={len(h['deaths']):3d}  births={len(h['births']):3d}  "
          f"time={h['times'][-1]:.1f}s")


In [ ]:
# ---- Live convergence plot from the light demo -------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(gmm_full["times"],    gmm_full["losses"],    label="Full",    color="green")
axes[0].plot(gmm_sto["times"],     gmm_sto["losses"],     label="Sto",     color="blue")
axes[0].plot(gmm_full_bd["times"], gmm_full_bd["losses"], label="Full+BD", color="limegreen", ls="--")
axes[0].plot(gmm_sto_bd["times"],  gmm_sto_bd["losses"],  label="Sto+BD",  color="dodgerblue", ls="--")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("BLASSO loss")
axes[0].set_title("Light demo - Loss vs time"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(gmm_full["times"],    gmm_full["tvs"],    label="Full",    color="green")
axes[1].plot(gmm_sto["times"],     gmm_sto["tvs"],     label="Sto",     color="blue")
axes[1].plot(gmm_full_bd["times"], gmm_full_bd["tvs"], label="Full+BD", color="limegreen", ls="--")
axes[1].plot(gmm_sto_bd["times"],  gmm_sto_bd["tvs"],  label="Sto+BD",  color="dodgerblue", ls="--")
axes[1].set_xscale("log"); axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("TV norm")
axes[1].set_title("Light demo - TV norm vs time"); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(gmm_full_bd["iters"], gmm_full_bd["ps"], label="Full+BD", color="limegreen")
axes[2].plot(gmm_sto_bd["iters"],  gmm_sto_bd["ps"],  label="Sto+BD",  color="dodgerblue")
axes[2].axhline(GMM_N_PARTICLES, color="grey", ls=":", label=f"$p_0={GMM_N_PARTICLES}$")
for it, p in gmm_full_bd["deaths"]: axes[2].plot(it, p, "v", color="red",   ms=6)
for it, p in gmm_full_bd["births"]: axes[2].plot(it, p, "^", color="green", ms=6)
for it, p in gmm_sto_bd["deaths"]:  axes[2].plot(it, p, "v", color="red",   ms=4, alpha=0.6)
for it, p in gmm_sto_bd["births"]:  axes[2].plot(it, p, "^", color="green", ms=4, alpha=0.6)
axes[2].set_xlabel("Iteration"); axes[2].set_ylabel("Particle count")
axes[2].set_title("Light demo - BD events"); axes[2].legend(); axes[2].grid(alpha=.3)

plt.suptitle("Section 4.1 GMM - light demo diagnostics", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Live final positions in R^2 --------------------------------------------

def _plot_positions(ax, particles, title, color):
    with torch.no_grad():
        pos = particles.positions.detach().cpu().numpy()
        a   = particles.amplitudes.detach().cpu().numpy()
    sz = 30 + 600 * a / (a.max() + 1e-12)
    ax.scatter(pos[:, 0], pos[:, 1], s=sz, color=color, alpha=0.85,
               edgecolors="k", linewidths=0.4, zorder=3)
    m_  = true_means.numpy()
    wn_ = (true_weights / true_weights.sum()).numpy()
    ax.scatter(m_[:, 0], m_[:, 1], s=20 + 800*wn_, marker="*",
               color="crimson", zorder=2, label="True means")
    ax.set_title(f"{title} (p={len(particles)})", fontsize=10)
    ax.set_aspect("equal"); ax.grid(alpha=.2)
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
_plot_positions(axes[0,0], gmm_full["particles"],    "Full-Batch (no BD)",         "steelblue")
_plot_positions(axes[0,1], gmm_sto["particles"],     "Stochastic (no BD)",         "purple")
_plot_positions(axes[1,0], gmm_full_bd["particles"], "Full-Batch + Birth-Death",   "seagreen")
_plot_positions(axes[1,1], gmm_sto_bd["particles"],  "Stochastic + Birth-Death",   "darkorange")
plt.suptitle("Section 4.1 GMM - final particle positions (light demo)", y=1.01)
plt.tight_layout(); plt.show()


### 2.5 Paper figures — GMM

The full-budget GMM run produces the figures referenced by §4.3 of the paper:
- `fig3a_loss_vs_time_bd.pdf` → left panel of the paper's Figure on convergence.
- `fig3e_bd_events_full.pdf` / `fig3f_bd_events_sto.pdf` → bottom row of the paper's Figure on BD dynamics.
- `fig4a_*` through `fig4d_*` → the 2×2 grid of the paper's Figure on final GMM positions.


In [ ]:
show_pdf(GMM_RESULTS / "fig3a_loss_vs_time_bd.pdf",
         width=620, height=440,
         caption="Paper Fig. (left panel of fig:loss_vs_time) -- BLASSO loss vs wall-clock time, 4-way.")

show_pdf_grid([
    (GMM_RESULTS / "fig3e_bd_events_full.pdf", "Paper Fig. (bottom-left of fig:bd_events) -- Full-Batch+BD timeline"),
    (GMM_RESULTS / "fig3f_bd_events_sto.pdf",  "Paper Fig. (bottom-right of fig:bd_events) -- Stochastic+BD timeline"),
], ncols=2, width=430, height=320)

show_pdf_grid([
    (GMM_RESULTS / "fig4a_positions_full.pdf",    "Paper Fig. fig:gmm_positions (top-left) -- Full, no BD"),
    (GMM_RESULTS / "fig4b_positions_sto.pdf",     "Paper Fig. fig:gmm_positions (top-right) -- Stoch, no BD"),
    (GMM_RESULTS / "fig4c_positions_full_bd.pdf", "Paper Fig. fig:gmm_positions (bottom-left) -- Full+BD"),
    (GMM_RESULTS / "fig4d_positions_sto_bd.pdf",  "Paper Fig. fig:gmm_positions (bottom-right) -- Stoch+BD"),
], ncols=2, width=430, height=320)


## 3. §4.2 Two-layer NN on California Housing

We move to a real-world regression task. Each "particle" is a ReLU neuron with parameters $\theta=(w,b)\in\bar B(0,1)\subset\mathbb{R}^{d+1}$ (positive homogeneity of ReLU pushes the locations onto the unit sphere). For California Housing $d=8$, $n=18\,576$ training samples, $\kappa=5\cdot 10^{-4}$.

The objective is the regularised MSE (paper Eq. (4.2)):
$$
J(\nu)\;=\;\tfrac{1}{2}\,\|y-\Phi(\nu)\|_{\mathbb{H}}^{2}\;+\;\kappa\,\|\nu\|_{\mathrm{TV}},
$$
with feature map $\varphi_{(w,b)}(x)=\mathrm{ReLU}(\langle w,x\rangle+b)$ and empirical inner product $\langle u,v\rangle_{\mathbb{H}}=\frac{1}{n}\sum_k u_k v_k$.

The kernel reads
$$
K(\theta_i,\theta_j)\;=\;\frac{1}{n}\sum_{k=1}^{n}\mathrm{ReLU}(\langle w_i,x_k\rangle + b_i)\,\mathrm{ReLU}(\langle w_j,x_k\rangle + b_j).
$$


### 3.1 ReLU and the $(\mathcal{H}_{\mathcal{P}})$ caveat

ReLU is Lipschitz but **not** $\mathcal{C}^{2}$, so the bounded $\mathcal{C}^{2}$ smoothness assumption $(\mathcal{H}_{\mathcal{P}})$ of §2.2 is not strictly satisfied. In practice we use the ReLU subgradient
$$
\partial_\theta\,\mathrm{ReLU}(\langle w,x\rangle+b)\;=\;\mathbf{1}\!\{\langle w,x\rangle+b>0\}\,(x,1),
$$
and the experiment shows that the algorithm remains effective despite the theoretical mismatch. This is consistent with the paper's qualitative finding.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---- Dataset --------------------------------------------------------------
housing = fetch_california_housing()
X_raw, y_raw = housing.data, housing.target
print(f"California Housing : {X_raw.shape[0]} samples, d={X_raw.shape[1]}")

Xtr_np, Xte_np, ytr_np, yte_np = train_test_split(
    X_raw, y_raw, test_size=0.1, random_state=314)
scaler = StandardScaler()
Xtr_np = scaler.fit_transform(Xtr_np)
Xte_np = scaler.transform(Xte_np)

X_train_nn = torch.tensor(Xtr_np, dtype=torch.float32)
y_train_nn = torch.tensor(ytr_np, dtype=torch.float32).unsqueeze(-1)
X_test_nn  = torch.tensor(Xte_np, dtype=torch.float32)
y_test_nn  = torch.tensor(yte_np, dtype=torch.float32).unsqueeze(-1)
print(f"Train: {X_train_nn.shape}, Test: {X_test_nn.shape}")


In [ ]:
# ============================================================================
# NN BLASSO and optimisers
# ============================================================================

@dataclass
class NNBlasso(L2Blasso[Tensor]):
    """ReLU-feature BLASSO: phi_{(w,b)}(x) = ReLU(<w,x>+b)."""
    inputs: Tensor
    R_X: float = 1.0

    def __post_init__(self):
        self.input_dim = self.inputs.shape[1]
        self.loss_offset = 0.5 * (self.y ** 2).mean().item()

    @dataclass
    class Evaluation(L2Blasso.Evaluation):
        @cached_property
        def phi(self):
            w = self.particles.positions[:, :-1]
            b = self.particles.positions[:, -1:]
            return torch.relu(w @ self.problem.inputs.T + b)

        @cached_property
        def prediction(self):
            a_signed = self.amplitudes * self.particles.signs
            return (a_signed[:, None] * self.phi).sum(0).unsqueeze(-1)

        @cached_property
        def kernel(self):
            return self.phi @ self.phi.T / self.phi.shape[1]

        @cached_property
        def scalar_products(self):
            return (self.phi @ self.problem.y).squeeze() / self.phi.shape[1]

        @cached_property
        def proximity(self):
            return 0.5 * ((self.problem.y - self.prediction) ** 2).mean()

        @cached_property
        def Jp(self):
            return (self.problem.kappa
                    + self.kernel @ (self.amplitudes * self.particles.signs)
                    - self.scalar_products)

        @cached_property
        def Jp_grad(self):
            relu_mask = (self.phi > 0).float()
            a_signed  = self.amplitudes * self.particles.signs
            hat_y     = (a_signed[:, None] * self.phi).sum(0)
            residual  = hat_y - self.problem.y.squeeze()
            grad_w    = (relu_mask * residual[None, :]) @ self.problem.inputs / relu_mask.shape[1]
            grad_b    = (relu_mask * residual[None, :]).sum(1, keepdim=True) / relu_mask.shape[1]
            return torch.cat([grad_w, grad_b], dim=1)


@dataclass
class NNOptimizer:
    """Full-batch CPGD for the NN (signed-measure, mirror retraction + unit-ball proj)."""
    particles: Particles
    problem:   NNBlasso
    eta:       float
    eta_w:     float
    min_weight: float = 1e-6

    def step(self):
        positions = self.particles.positions
        weights   = self.particles.weights
        signs     = self.particles.signs
        ev = self.problem(self.particles)
        with torch.no_grad():
            jp      = ev.Jp.clone()
            jp_grad = ev.Jp_grad.clone()
            grad_omega = signs * jp + self.problem.kappa * (1.0 - signs)
            weights.mul_(torch.exp(-self.eta_w * grad_omega))
            weights.clamp_(min=self.min_weight)
            g = signs.unsqueeze(-1) * jp_grad
            positions.sub_(self.eta * g)
            positions.div_(positions.norm(dim=1, keepdim=True).clamp(min=1.0))
        return ev.loss.item()

    def zero_grad(self): self.particles.zero_grad()


@dataclass
class StochasticNNOptimizer:
    particles:  Particles
    problem:    NNBlasso
    eta:        float
    eta_w:      float
    batch_size: int
    min_weight: float = 1e-9

    def __post_init__(self):
        self.n_samples = len(self.problem.inputs)

    def step(self):
        idx = torch.randperm(self.n_samples)[:self.batch_size]
        mini = NNBlasso(y=self.problem.y[idx], inputs=self.problem.inputs[idx],
                        kappa=self.problem.kappa, R_X=self.problem.R_X)
        ev = mini(self.particles)
        positions, weights, signs = self.particles.positions, self.particles.weights, self.particles.signs
        with torch.no_grad():
            jp = ev.Jp.clone()
            jp_grad = ev.Jp_grad.clone()
            grad_omega = signs * jp + self.problem.kappa * (1.0 - signs)
            weights.mul_(torch.exp(-self.eta_w * grad_omega))
            weights.clamp_(min=self.min_weight)
            g = signs.unsqueeze(-1) * jp_grad
            positions.sub_(self.eta * g)
            positions.div_(positions.norm(dim=1, keepdim=True).clamp(min=1.0))
        return ev.loss.item()

    def zero_grad(self): self.particles.zero_grad()


# ---- NN Birth-Death --------------------------------------------------------

def _bd_remove_nn(particles, idx):
    keep = [i for i in range(len(particles)) if i != idx]
    with torch.no_grad():
        return Particles(particles.h,
                         particles.positions.data[keep].clone(),
                         particles.weights.data[keep].clone(),
                         particles.signs[keep].clone())

def _bd_add_nn(particles, new_pos, new_w, new_sign=+1.0):
    with torch.no_grad():
        pos = torch.cat([particles.positions.data, new_pos.unsqueeze(0)], 0)
        w   = torch.cat([particles.weights.data,   torch.tensor([new_w])], 0)
        s   = torch.cat([particles.signs,          torch.tensor([new_sign])], 0)
    return Particles(particles.h, pos, w, s)


def death_step_nn(particles, problem, threshold):
    with torch.no_grad():
        ev      = problem(particles)
        K_diag  = ev.phi.pow(2).sum(1) / ev.phi.shape[1]
        scores  = 2.0 * particles.signs * ev.Jp / (particles.weights * K_diag.clamp(min=1e-10))
        mv, mi  = scores.max(0)
        if mv.item() > threshold:
            return _bd_remove_nn(particles, mi.item()), True
    return particles, False


def birth_step_nn(particles, problem, n_test, threshold, theta_dim):
    with torch.no_grad():
        test_pos = torch.randn(n_test, theta_dim)
        test_pos = test_pos / test_pos.norm(dim=1, keepdim=True)
        w_t, b_t = test_pos[:, :-1], test_pos[:, -1:]
        phi_t = torch.relu(w_t @ problem.inputs.T + b_t)
        ev_curr = problem(particles)
        a_signed = particles.amplitudes * particles.signs
        phi_curr = ev_curr.phi
        n_samples = phi_curr.shape[1]
        K_test  = phi_t @ phi_curr.T / n_samples
        sp_test = (phi_t @ problem.y).squeeze() / n_samples
        Jp_test = problem.kappa + K_test @ a_signed - sp_test
        mv, mi  = Jp_test.min(0)
        if mv.item() < threshold:
            phi_new = phi_t[mi.item()]
            K_self  = phi_new.pow(2).sum() / n_samples
            w0 = (mv.clamp(max=1).abs() / K_self.clamp(min=1e-3)).item()
            return _bd_add_nn(particles, test_pos[mi.item()], w0, +1.0), True
    return particles, False

print("NN classes (NNBlasso, NNOptimizer, StochasticNNOptimizer) and BD helpers loaded.")


### 3.2 Hyperparameters (paper Table 1, NN column)

| Symbol | Paper value | Light-demo value |
|---|---|---|
| $n$ (train) | $18\,576$ | $18\,576$ |
| $p_{\mathrm{init}}$ | $300$ | $300$ |
| $\kappa$ | $5\cdot 10^{-4}$ | $5\cdot 10^{-4}$ |
| Full-batch iter. | $100\,000$ | $5\,000$ |
| Stoch. iter. | $750\,000$ | $30\,000$ |
| Stoch. batch | $256$ | $256$ |
| $\tau_{\mathrm{death}}$ | $5$ | $5$ |
| $\eta$ (position) / $\eta_w$ (weight) | $10^{-2}$ / $10^{-1}$ | $10^{-2}$ / $10^{-1}$ |


In [ ]:
# ---- NN Hyperparameters -----------------------------------------------------
NN_KAPPA          = 5e-4
NN_N_PARTICLES    = 300
NN_ETA            = 1e-2
NN_ETA_W          = 1e-1
NN_BATCH_SIZE     = 256
NN_INPUT_DIM      = X_train_nn.shape[1]
NN_THETA_DIM      = NN_INPUT_DIM + 1

# Light demo budget (paper values: 100_000 / 750_000)
NN_T_FULL = 5_000
NN_T_STO  = 30_000
NN_LOG    = 500

BD_NN_FULL_DEATH_DELAY = 1_000
BD_NN_FULL_DEATH_EVERY = 200
BD_NN_FULL_BIRTH_DELAY = 500
BD_NN_FULL_BIRTH_EVERY = 400
BD_NN_STO_DEATH_DELAY  = 5_000
BD_NN_STO_DEATH_EVERY  = 500
BD_NN_STO_BIRTH_DELAY  = 3_000
BD_NN_STO_BIRTH_EVERY  = 2_000
BD_NN_N_TEST           = 1_000

BD_NN_TAU_BIRTH_FULL = -1.0 * math.sqrt(NN_THETA_DIM * math.log(len(X_train_nn)) / len(X_train_nn))
BD_NN_TAU_BIRTH_STO  =  2.0 * math.sqrt(NN_THETA_DIM * math.log(NN_BATCH_SIZE) / NN_BATCH_SIZE)

nn_problem      = NNBlasso(kappa=NN_KAPPA, y=y_train_nn, inputs=X_train_nn, R_X=1.0)
nn_problem_test = NNBlasso(kappa=NN_KAPPA, y=y_test_nn,  inputs=X_test_nn,  R_X=1.0)


### 3.3 Light demo — four methods

Note: the NN demo is the most expensive part (matrix mults of size $p\times n$). On CPU expect ~1–2 min for the full sweep at the demo budget.


In [ ]:
def init_nn_particles(n, theta_dim, seed):
    torch.manual_seed(seed)
    pos = torch.randn(n, theta_dim)
    pos = pos / pos.norm(dim=1, keepdim=True)
    w   = (10.0 / n) * torch.ones(n)
    s   = torch.ones(n)
    return Particles(h=lambda r: r, positions=pos, weights=w, signs=s)


def run_nn_method(method, T):
    p0 = init_nn_particles(NN_N_PARTICLES, NN_THETA_DIM, seed=314)
    if method == "full":
        opt = NNOptimizer(p0, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W); bd = None
    elif method == "sto":
        opt = StochasticNNOptimizer(p0, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W,
                                    batch_size=NN_BATCH_SIZE); bd = None
    elif method == "full_bd":
        opt = NNOptimizer(p0, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W)
        bd = dict(kind="full", death_delay=BD_NN_FULL_DEATH_DELAY, death_every=BD_NN_FULL_DEATH_EVERY,
                  birth_delay=BD_NN_FULL_BIRTH_DELAY, birth_every=BD_NN_FULL_BIRTH_EVERY,
                  tau_birth=BD_NN_TAU_BIRTH_FULL)
    elif method == "sto_bd":
        opt = StochasticNNOptimizer(p0, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W,
                                    batch_size=NN_BATCH_SIZE)
        bd = dict(kind="sto", death_delay=BD_NN_STO_DEATH_DELAY, death_every=BD_NN_STO_DEATH_EVERY,
                  birth_delay=BD_NN_STO_BIRTH_DELAY, birth_every=BD_NN_STO_BIRTH_EVERY,
                  tau_birth=BD_NN_TAU_BIRTH_STO)
    losses, mses_tr, mses_te, tvs, times, ps, iters = [], [], [], [], [], [], []
    deaths, births = [], []
    t0 = time.time()
    for it in range(T):
        opt.zero_grad(); opt.step()
        if bd is not None:
            if it >= bd["death_delay"] and it % bd["death_every"] == 0 and len(opt.particles) > 1:
                newp, died = death_step_nn(opt.particles, nn_problem, BD_TAU_DEATH)
                if died:
                    deaths.append((it, len(newp)))
                    if bd["kind"] == "full":
                        opt = NNOptimizer(newp, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W)
                    else:
                        opt = StochasticNNOptimizer(newp, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W,
                                                    batch_size=NN_BATCH_SIZE)
            if it >= bd["birth_delay"] and it % bd["birth_every"] == 0:
                newp, born = birth_step_nn(opt.particles, nn_problem,
                                           BD_NN_N_TEST, bd["tau_birth"], NN_THETA_DIM)
                if born:
                    births.append((it, len(newp)))
                    if bd["kind"] == "full":
                        opt = NNOptimizer(newp, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W)
                    else:
                        opt = StochasticNNOptimizer(newp, nn_problem, eta=NN_ETA, eta_w=NN_ETA_W,
                                                    batch_size=NN_BATCH_SIZE)
        if it % NN_LOG == 0 or it == T-1:
            with torch.no_grad():
                ev_tr = nn_problem(opt.particles)
                ev_te = nn_problem_test(opt.particles)
                losses.append(ev_tr.loss.item())
                mses_tr.append(ev_tr.proximity.item() * 2)
                mses_te.append(ev_te.proximity.item() * 2)
                tvs.append(opt.particles.tv_norm().item())
                ps.append(len(opt.particles))
                iters.append(it)
                times.append(time.time() - t0)
    return dict(particles=opt.particles, losses=losses, mses_tr=mses_tr, mses_te=mses_te,
                tvs=tvs, times=times, ps=ps, iters=iters, deaths=deaths, births=births, T=T)


print("Running NN light demo (4 methods)...")
t_start = time.time()
nn_full     = run_nn_method("full",    NN_T_FULL)
nn_sto      = run_nn_method("sto",     NN_T_STO)
nn_full_bd  = run_nn_method("full_bd", NN_T_FULL)
nn_sto_bd   = run_nn_method("sto_bd",  NN_T_STO)
print(f"\nNN demo finished in {time.time()-t_start:.1f}s.")

for name, h in [("Full", nn_full), ("Sto", nn_sto), ("Full+BD", nn_full_bd), ("Sto+BD", nn_sto_bd)]:
    print(f"  {name:<10s}  loss={h['losses'][-1]:.5f}  MSE_tr={h['mses_tr'][-1]:.4f}  "
          f"MSE_te={h['mses_te'][-1]:.4f}  TV={h['tvs'][-1]:.3f}  "
          f"p_final={h['ps'][-1]:3d}  D={len(h['deaths']):3d}  B={len(h['births']):3d}  "
          f"time={h['times'][-1]:.1f}s")


In [ ]:
# ---- Live NN convergence ----------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(nn_full["times"],    nn_full["losses"],    label="Full",    color="green")
axes[0].plot(nn_sto["times"],     nn_sto["losses"],     label="Sto",     color="blue")
axes[0].plot(nn_full_bd["times"], nn_full_bd["losses"], label="Full+BD", color="limegreen", ls="--")
axes[0].plot(nn_sto_bd["times"],  nn_sto_bd["losses"],  label="Sto+BD",  color="dodgerblue", ls="--")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("BLASSO loss")
axes[0].set_title("NN demo - Loss vs time"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(nn_full["times"],    nn_full["mses_te"],    label="Full",    color="green")
axes[1].plot(nn_sto["times"],     nn_sto["mses_te"],     label="Sto",     color="blue")
axes[1].plot(nn_full_bd["times"], nn_full_bd["mses_te"], label="Full+BD", color="limegreen", ls="--")
axes[1].plot(nn_sto_bd["times"],  nn_sto_bd["mses_te"],  label="Sto+BD",  color="dodgerblue", ls="--")
axes[1].set_xscale("log"); axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Test MSE")
axes[1].set_title("NN demo - Test MSE vs time"); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(nn_full_bd["iters"], nn_full_bd["ps"], label="Full+BD", color="limegreen")
axes[2].plot(nn_sto_bd["iters"],  nn_sto_bd["ps"],  label="Sto+BD",  color="dodgerblue")
axes[2].axhline(NN_N_PARTICLES, color="grey", ls=":", label=f"$p_0={NN_N_PARTICLES}$")
for it, p in nn_full_bd["deaths"]: axes[2].plot(it, p, "v", color="red",   ms=6)
for it, p in nn_full_bd["births"]: axes[2].plot(it, p, "^", color="green", ms=6)
for it, p in nn_sto_bd["deaths"]:  axes[2].plot(it, p, "v", color="red",   ms=4, alpha=0.6)
for it, p in nn_sto_bd["births"]:  axes[2].plot(it, p, "^", color="green", ms=4, alpha=0.6)
axes[2].set_xlabel("Iteration"); axes[2].set_ylabel("Particle count")
axes[2].set_title("NN demo - BD events"); axes[2].legend(); axes[2].grid(alpha=.3)

plt.suptitle("Section 4.2 NN - light demo diagnostics", y=1.02)
plt.tight_layout(); plt.show()


### 3.4 Paper figures — NN

These are the NN figures *that appear in the paper*:
- `fig1a_loss_vs_time.pdf` / `fig2a_loss_vs_time_bd.pdf` → right panel of the paper's Figure on convergence.
- `fig2e_bd_events_full.pdf` / `fig2f_bd_events_sto.pdf` → top row of the paper's Figure on BD dynamics.


In [ ]:
show_pdf(NN_RESULTS / "fig2a_loss_vs_time_bd.pdf",
         width=620, height=440,
         caption="Paper Fig. (right panel of fig:loss_vs_time) -- BLASSO loss vs wall-clock time, 4-way.")

show_pdf_grid([
    (NN_RESULTS / "fig2e_bd_events_full.pdf", "Paper Fig. (top-left of fig:bd_events) -- Full+BD timeline"),
    (NN_RESULTS / "fig2f_bd_events_sto.pdf",  "Paper Fig. (top-right of fig:bd_events) -- Sto+BD timeline"),
], ncols=2, width=430, height=320)


### 3.5 Companion-only NN figures (not in paper)

The NN run produces three additional families of diagnostic plots that did not make it into Section 4 of the paper. They are nonetheless very informative and are included here as pedagogical companions:

- `fig3a-d_dual_cert_*.pdf` — scatter of $(\omega_i,\eta(\theta_i))$ where $\eta(\theta_i)=1-J'_\nu(\theta_i)/\kappa$ is the normalised dual certificate. KKT predicts $\eta(\theta_i)=+1$ at every active particle.
- `fig4a-d_weights_*.pdf` — sorted bar charts of $\omega_i$ on log scale (sparsity diagnostic).
- `fig5*_positions_*.pdf` — 2D projections of the particle positions on $\mathbb{S}^{d}$ via random, max-energy or PCA projections.


In [ ]:
# ---- Dual certificate (companion-only) -------------------------------------
show_pdf_grid([
    (NN_RESULTS / "fig3a_dual_cert_sto.pdf",     "Stoch (no BD) - dual certificate vs omega"),
    (NN_RESULTS / "fig3b_dual_cert_full.pdf",    "Full (no BD) - dual certificate vs omega"),
    (NN_RESULTS / "fig3c_dual_cert_sto_bd.pdf",  "Stoch + BD - dual certificate vs omega"),
    (NN_RESULTS / "fig3d_dual_cert_full_bd.pdf", "Full + BD - dual certificate vs omega"),
], ncols=2, width=430, height=320)


In [ ]:
# ---- Particle weights (companion-only) -------------------------------------
show_pdf_grid([
    (NN_RESULTS / "fig4a_weights_sto.pdf",     "Stoch (no BD) - sorted weights"),
    (NN_RESULTS / "fig4b_weights_full.pdf",    "Full (no BD) - sorted weights"),
    (NN_RESULTS / "fig4c_weights_sto_bd.pdf",  "Stoch + BD - sorted weights"),
    (NN_RESULTS / "fig4d_weights_full_bd.pdf", "Full + BD - sorted weights"),
], ncols=2, width=430, height=320)


In [ ]:
# ---- Particle positions on the sphere (companion-only) ---------------------
show_pdf_grid([
    (NN_RESULTS / "fig5a_pos_pca_sto.pdf",     "Stoch (no BD) - PCA 2D projection"),
    (NN_RESULTS / "fig5b_pos_pca_full.pdf",    "Full (no BD) - PCA 2D projection"),
    (NN_RESULTS / "fig5c_pos_pca_sto_bd.pdf",  "Stoch + BD - PCA 2D projection"),
    (NN_RESULTS / "fig5d_pos_pca_full_bd.pdf", "Full + BD - PCA 2D projection"),
], ncols=2, width=430, height=320)


## 4. §4.3 Experimental Results & Discussion

This section assembles the cross-experiment figures and tables that drive the paper's discussion.


### 4.1 §4.3.1 Dynamics of the Birth-Death process

Particle count over iterations for both experiments. Top row: Neural-Network on California. Bottom row: GMM. Sharp downward spikes correspond to **death events** (redundant particles pruned); sharp upward spikes correspond to **birth events** (new particles spawned by the birth process). The neural-network capacity gets reduced by **50-70%** under BD; the GMM example stabilises around $p=39$ (Stochastic+BD, right) and $p=10$ (Full+BD, left), against the initial $p=20$.


In [ ]:
# Paper Figure fig:bd_events (main_jmlr.tex:1241) - 2x2 grid

show_pdf_grid([
    (NN_RESULTS / "fig2e_bd_events_full.pdf",  "NN: Full + BD (top-left)"),
    (NN_RESULTS / "fig2f_bd_events_sto.pdf",   "NN: Stochastic + BD (top-right)"),
    (GMM_RESULTS / "fig3e_bd_events_full.pdf", "GMM: Full + BD (bottom-left)"),
    (GMM_RESULTS / "fig3f_bd_events_sto.pdf",  "GMM: Stochastic + BD (bottom-right)"),
], ncols=2, width=430, height=320)


### 4.2 §4.3.2 Convergence and generalisation

Two paper panels of the convergence figure: the GMM on the left, the NN on the right. The standard CPGD methods *plateau into local minima*; the BD-augmented variants periodically inject new particles in regions where the dual certificate is strongly negative, *allowing the loss to drop further*. On the NN side, the solution gets sparser while achieving the same test performance as the over-parametrised baseline.


In [ ]:
show_pdf_grid([
    (GMM_RESULTS / "fig3a_loss_vs_time_bd.pdf", "GMM (paper fig:loss_vs_time, left)"),
    (NN_RESULTS  / "fig2a_loss_vs_time_bd.pdf", "NN  (paper fig:loss_vs_time, right)"),
], ncols=2, width=480, height=340)


### 4.3 §4.3.3 Spatial distribution of GMM particles

Final particle positions for the four GMM methods. The top row (no BD) keeps **spurious particles** that fail to align with the true means; the bottom row (BD) cleans them up. Full+BD ends with $p=10$ (under-representation); Stochastic+BD ends with $p=39$ and identifies **all targets except the smallest one** (weight $\approx 5\times 10^{-4}$).


In [ ]:
show_pdf_grid([
    (GMM_RESULTS / "fig4a_positions_full.pdf",    "Top-left:  Full,        no BD"),
    (GMM_RESULTS / "fig4b_positions_sto.pdf",     "Top-right: Stochastic,  no BD"),
    (GMM_RESULTS / "fig4c_positions_full_bd.pdf", "Bottom-left:  Full       + BD"),
    (GMM_RESULTS / "fig4d_positions_sto_bd.pdf",  "Bottom-right: Stochastic + BD"),
], ncols=2, width=430, height=320)


### 4.4 §4.3.4 Performance Analysis

We display the paper's two final tables (Table 2 of Section 4.3.4 for the GMM, Table 3 of Section 4.3.4 for the NN), then for transparency we report the same metrics from the light demo above so the reader can see how the qualitative ranking (BD wins on both axes) survives a much shorter optimisation budget.


In [ ]:
# ---- Paper Table tab:gmm_results (main_jmlr.tex:1325) ----------------------

paper_gmm = [
    # method,            loss,     TV,     p_final, time, deaths, births
    ("Full-Batch",       0.001869, 0.2760, 20, 83.67, 0,  0),
    ("Stochastic",       0.001872, 0.2863, 20, 55.02, 0,  0),
    ("Full-Batch + BD",  0.000434, 0.7788, 10, 61.88, 19, 9),
    ("Stochastic + BD",  0.000259, 0.9786, 39, 65.23, 78, 97),
]

print("PAPER Table 2 (GMM) - main_jmlr.tex:1325")
print(f"{'Method':<22} {'Loss':>10} {'TV':>8} {'p_final':>9} {'Time(s)':>9} {'D':>5} {'B':>5}")
print("-"*70)
for m, l, tv, p, t, d, b in paper_gmm:
    bold = "*" if "+ BD" in m and (l < 0.0005 or tv > 0.97) else " "
    print(f"{m:<22} {l:>10.6f} {tv:>8.4f} {p:>9d} {t:>9.2f} {d:>5d} {b:>5d} {bold}")


In [ ]:
# ---- Paper Table tab:nn_results (main_jmlr.tex:1348) -----------------------

paper_nn = [
    # method,            MSE_tr,    MSE_te,    p_final, time(s), deaths, births
    ("Full-Batch",       0.365476, 0.393433, 300, 892.2, 0,   0),
    ("Stochastic",       0.366231, 0.393154, 300, 265.1, 0,   0),
    ("Full-Batch + BD",  0.365506, 0.393495, 189, 644.0, 111, 0),
    ("Stochastic + BD",  0.366369, 0.392429,  60, 158.9, 269, 29),
]

print("PAPER Table 3 (NN California) - main_jmlr.tex:1348")
print(f"{'Method':<22} {'MSE(tr)':>10} {'MSE(te)':>10} {'p_final':>9} {'Time(s)':>9} {'D':>5} {'B':>5}")
print("-"*72)
for m, mtr, mte, p, t, d, b in paper_nn:
    bold = "*" if "+ BD" in m else " "
    print(f"{m:<22} {mtr:>10.6f} {mte:>10.6f} {p:>9d} {t:>9.1f} {d:>5d} {b:>5d} {bold}")


In [ ]:
# ---- Light-demo metrics tables (NOT paper numbers) -------------------------

print("LIGHT-DEMO Table - GMM (5k / 20k iterations - NOT paper budget)")
print(f"{'Method':<22} {'Loss':>10} {'TV':>8} {'p_final':>9} {'Time(s)':>9} {'D':>5} {'B':>5}")
print("-"*70)
for name, h in [("Full-Batch (demo)", gmm_full),
                ("Stochastic (demo)", gmm_sto),
                ("Full-Batch+BD (demo)", gmm_full_bd),
                ("Stochastic+BD (demo)", gmm_sto_bd)]:
    print(f"{name:<22} {h['losses'][-1]:>10.6f} {h['tvs'][-1]:>8.4f} "
          f"{h['ps'][-1]:>9d} {h['times'][-1]:>9.2f} "
          f"{len(h['deaths']):>5d} {len(h['births']):>5d}")

print()
print("LIGHT-DEMO Table - NN California (5k / 30k iterations - NOT paper budget)")
print(f"{'Method':<22} {'MSE(tr)':>10} {'MSE(te)':>10} {'p_final':>9} {'Time(s)':>9} {'D':>5} {'B':>5}")
print("-"*72)
for name, h in [("Full-Batch (demo)", nn_full),
                ("Stochastic (demo)", nn_sto),
                ("Full-Batch+BD (demo)", nn_full_bd),
                ("Stochastic+BD (demo)", nn_sto_bd)]:
    print(f"{name:<22} {h['mses_tr'][-1]:>10.6f} {h['mses_te'][-1]:>10.6f} "
          f"{h['ps'][-1]:>9d} {h['times'][-1]:>9.1f} "
          f"{len(h['deaths']):>5d} {len(h['births']):>5d}")


In [ ]:
# ---- Side-by-side bar chart: p_final, time, deaths, births  -----------------

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
methods = ["Full", "Sto", "Full+BD", "Sto+BD"]
colors  = ["green", "blue", "limegreen", "dodgerblue"]

# Paper numbers (Tables 2 & 3)
paper_gmm_p     = [20,   20,   10,   39]
paper_nn_p      = [300,  300,  189,  60]
paper_gmm_time  = [83.67, 55.02, 61.88, 65.23]
paper_nn_time   = [892.2, 265.1, 644.0, 158.9]
paper_gmm_DB    = [(0,0), (0,0), (19,9), (78,97)]
paper_nn_DB     = [(0,0), (0,0), (111,0), (269,29)]

x = np.arange(4)
w = 0.35

# p_final
ax = axes[0]
ax.bar(x - w/2, paper_gmm_p, w, label="GMM (paper)", color="seagreen")
ax.bar(x + w/2, paper_nn_p,  w, label="NN (paper)",  color="goldenrod")
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20)
ax.set_title("Final particle count $p_{final}$"); ax.legend(); ax.grid(alpha=.3, axis="y")

# Time
ax = axes[1]
ax.bar(x - w/2, paper_gmm_time, w, label="GMM (paper)", color="seagreen")
ax.bar(x + w/2, paper_nn_time,  w, label="NN (paper)",  color="goldenrod")
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20)
ax.set_title("Wall-clock time (s)"); ax.set_yscale("log"); ax.legend(); ax.grid(alpha=.3, axis="y")

# Deaths
ax = axes[2]
ax.bar(x - w/2, [d for d,b in paper_gmm_DB], w, label="GMM (paper)", color="firebrick")
ax.bar(x + w/2, [d for d,b in paper_nn_DB],  w, label="NN (paper)",  color="indianred")
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20)
ax.set_title("Total deaths"); ax.legend(); ax.grid(alpha=.3, axis="y")

# Births
ax = axes[3]
ax.bar(x - w/2, [b for d,b in paper_gmm_DB], w, label="GMM (paper)", color="forestgreen")
ax.bar(x + w/2, [b for d,b in paper_nn_DB],  w, label="NN (paper)",  color="limegreen")
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20)
ax.set_title("Total births"); ax.legend(); ax.grid(alpha=.3, axis="y")

plt.suptitle("Section 4.3.4 - Paper Tables 2 & 3 in bar form", y=1.04)
plt.tight_layout(); plt.show()


## 5. Math appendix

Self-contained derivations used by the algorithm but cross-referenced to the paper for proofs.


### 5.1 Symmetrisation trick (signed → positive measures)

The BLASSO problem in §2.1 is posed over the space of **signed** Radon measures $\mathcal{M}(\mathcal{X})$. Standard practice (see e.g. Chizat 2022) is to lift it to the space $\mathcal{M}_+(\widetilde{\mathcal{X}})$ of **positive** measures on the doubled space $\widetilde{\mathcal{X}}=\mathcal{X}\times\{\pm 1\}$, where each "particle" carries both a position $t\in\mathcal{X}$ and a fixed sign $\varepsilon\in\{\pm 1\}$. The BLASSO loss is invariant under this lifting, and a positive measure on $\widetilde{\mathcal{X}}$ projects back to a signed measure on $\mathcal{X}$ via $\nu\mapsto \varepsilon\,\nu_{|\mathcal{X}\times\{\varepsilon\}}$.

In our code:
- GMM: all signs are $+1$ (the underlying density is non-negative).
- NN: all signs are $+1$ because antipodal positions on $\mathbb{S}^{d}$ with sign $+1$ are equivalent to the same positions with sign $-1$ — the ReLU positive homogeneity plus the antipodal symmetry of $\mathbb{S}^{d}$ covers the full signed setting.


### 5.2 Mirror retraction & weight reparametrisation

To keep $\omega_j>0$ along the trajectory we use one of two parametrisations:

**(a) Polynomial reparametrisation** $\omega = h(r) = r^{2}$ (GMM). The descent direction in $a$-space is recovered by dividing by $h'(r)=2r$:
$$
r \;\leftarrow\; r - \eta_w\,\frac{1}{h'(r)}\,\partial_r\!\left[\kappa\|\nu\|_{\mathrm{TV}} + \tfrac{1}{2}a^\top K a - a^\top S\right] \;=\; r - \eta_w\,J'_\nu(t).
$$

**(b) Mirror retraction** $\omega \leftarrow \omega\,e^{-\eta_w J'_\nu(t)}$ (NN). This is the exponentiated-gradient update on the positive orthant; it preserves $\omega>0$ automatically and corresponds to the *Push-Forward* step of §2.3 with the entropy mirror map.

Both are equivalent up to a change of step-size schedule.


### 5.3 First-order optimality and dual certificate

For $\nu = \sum_j \omega_j \delta_{t_j}$ the Fréchet derivative of $J$ along Dirac perturbations is
$$
J'_\nu(t)\;=\;\kappa\;+\;\sum_j \omega_j\, K(t,t_j)\;-\;\langle\varphi_t,y\rangle_{\mathbb{H}}.
$$
The first-order optimality condition for $\nu^\star$ (a positive minimiser) is
$$
\forall t\in\mathcal{X},\quad J'_{\nu^\star}(t)\;\ge\;0,\qquad
\text{and}\qquad J'_{\nu^\star}(t_j)\;=\;0 \quad \forall t_j\in\mathrm{supp}(\nu^\star).
$$
The **normalised dual certificate** $\eta(t) \;=\; 1 - J'_\nu(t)/\kappa$ satisfies the KKT conditions $\eta(t)\le 1$ everywhere and $\eta(t_j)=1$ at active particles. The scatter plots `fig3*_dual_cert_*.pdf` of §3.5 visualise this directly.


### 5.4 Birth-Death score derivation

**Death.** Compare $J(\nu)$ and $J(\nu - \omega_j\delta_{t_j})$ by exact second-order Taylor expansion in the amplitude. Letting $a=\omega_j$, $K_{jj}=K(t_j,t_j)$ and $J'_j=J'_\nu(t_j)$:
$$
J(\nu - a\,\delta_{t_j}) - J(\nu) \;=\; -a\,J'_j \;+\; \tfrac{1}{2}\,a^{2}\,K_{jj}.
$$
The right-hand side is **negative** (removal is beneficial) iff $a J'_j > \tfrac{1}{2}a^{2}K_{jj}$, equivalently
$$
\frac{2\,J'_j}{\omega_j\,K_{jj}}\;>\;1.
$$
Replacing the threshold $1$ by a safety factor $\tau_{\mathrm{death}}>1$ gives the death rule $s_j>\tau_{\mathrm{death}}$ used in §1.3.

**Birth.** Conversely, $J(\nu + a\delta_{t}) - J(\nu) = a J'_\nu(t) + \tfrac{1}{2}a^{2}K(t,t)$. Optimising over $a>0$ yields $a^\star = -J'_\nu(t)/K(t,t)$ when $J'_\nu(t)<0$, with optimal decrease $-\tfrac{1}{2}|J'_\nu(t)|^{2}/K(t,t)$. Hence the birth rule "$\min_k J'_\nu(\widehat{t}^{(k)}) < \tau_{\mathrm{birth}}$" exactly looks for the deepest dual-feasibility violation; the stochastic threshold $\tau_{\mathrm{birth}}\sqrt{\log m_k/m_k}$ accounts for the mini-batch noise on $J'_\nu$.


## How to reproduce paper numbers

To regenerate the saved `results_*` directories with the **full paper budget**, run:

- [experiments_fastpart_gmm_fixed_covariance.ipynb](experiments_fastpart_gmm_fixed_covariance.ipynb) — sets $T_{\text{full}}=50{,}000$, $T_{\text{sto}}=200{,}000$, $p_0=20$.
- [experiments_fastpart_nn_california.ipynb](experiments_fastpart_nn_california.ipynb) — sets $T_{\text{full}}=100{,}000$, $T_{\text{sto}}=750{,}000$, $p_0=300$.

Both notebooks dump their figures and `parameters.txt` into a timestamped directory `results_*` next to themselves. The companion notebook then picks them up via the `GMM_RESULTS` and `NN_RESULTS` paths set in the first code cell.

For deeper context on any term, refer to the corresponding part of the paper:
- BLASSO problem — Section 2.1
- First-order optimality — Section 2.2
- Weight & push-forward update — Section 2.3
- Fast Spawn & Prune algorithm — Section 3.2
- Numerical experiments — Section 4
- Symmetrisation trick — Appendix A.4 
